# MarketCast Finance Dashboard — Full Data Exploration

**Author:** Siham Chowdhury  
**Purpose:** Complete exploration of all tables feeding the finance dashboard. Documents the data model, calculation methodology, and unresolved questions. This notebook is the single source of truth for understanding the data before the 29 April meeting.

---

## Tables covered

| Table | Schema | Source system | What it contains |
|---|---|---|---|
| `rpt_project_revenue_and_costs` | `core` | NetSuite | Every financial transaction line — revenue, COGS, labour |
| `dim_customers` | `core` | NetSuite | Customer hierarchy — maps subsidiaries to top-level parents |
| `fact_sd_timesheet_cost` | `core` | Screendragon | Daily timesheet entries — people costs by project, role, day |
| `rpt_hubspot_line_report` | `core` | HubSpot | Pipeline deal lines |
| `fact_hubspot_targets` | `core` | HubSpot | Quarterly revenue targets by person and team |

## Data flow in the dashboard

```
rpt_project_revenue_and_costs  ──┐
                                  ├──► 5_gross_margin.sql ──► app.py KPIs + charts
dim_customers (join)          ──┘

fact_sd_timesheet_cost         ──► Labour classification ──► Labour tab
                                    (Research / Ops / Other)

rpt_hubspot_line_report        ──► Pipeline tab

fact_hubspot_targets           ──► Targets tab
```

## P&L calculation flow

```
Revenue         = -SUM(amount) WHERE account_type_id = 'Income'
COGS            =  SUM(amount) WHERE account_type_id = 'COGS'
Labour          =  SUM(amount) WHERE account_type_id IS NULL  ← timesheet rows

BE Allocation   = (customer BE rev / total BE rev) × $10,158,000 / 12  [2025]
AE Allocation   = (customer AE rev / total AE rev) × $938,000   / 12
RTA Allocation  = (customer RTA rev / total RTA rev) × $833,000 / 12

Gross Margin    = Revenue - COGS - BE Allocation - AE Allocation - RTA Allocation
GM%             = Gross Margin / Revenue × 100

Contribution    = Gross Margin - Labour
CM%             = Contribution / Revenue × 100
```

---

## 0. Setup

Connect to the Azure PostgreSQL data warehouse. The `get_engine()` function reads credentials from `.streamlit/secrets.toml` locally.

In [60]:
import sys
sys.path.insert(0, '/Users/sihamchowdhury/Desktop/Projects/finance-dashboards')

import pandas as pd
import numpy as np
from sqlalchemy import text
from src.db.connection import get_engine

engine = get_engine()

# Helper — run SQL and return DataFrame
def sql(query, **kwargs):
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

print("Connected to DB")

Connected to DB


---
## 1. Schema Overview

The DB has 203 tables across multiple schemas. We only use `core` — the analytics-ready layer built on top of NetSuite and HubSpot by the data engineering team.

In [61]:
# All tables in the core schema
sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'core'
    AND table_type = 'BASE TABLE'
    ORDER BY table_name
""")

,table_name
0,bridge_deal_line
1,dim_accounting_periods
2,dim_accounts
3,dim_classes
4,dim_consolidated_exchange_rates
...,...
73,stg_sd_user_role
74,stg_sd_user_role_snapshot
75,stg_sharepoint_customer_master
76,stg_sharepoint_services_master


---
## 2. `core.rpt_project_revenue_and_costs` — Main Fact Table

### What it is
This is the spine of the entire P&L. Every row is a **financial transaction line** from NetSuite — either a revenue recognition entry, a vendor bill (COGS), or a timesheet allocation (labour).

### How rows are typed
The `account_type_id` column tells you what kind of row you have:

| `account_type_id` | Meaning | How to treat |
|---|---|---|
| `'Income'` | Revenue | Negate `amount` — stored as negative (double-entry credits) |
| `'COGS'` | Cost of Sales | Sum `amount` directly |
| `NULL` | Labour / timesheet | Sum `amount` directly |

### Critical design decisions
1. **Use `amount` not `amount_usd`** — verified identical for all years
2. **Always filter `accounting_period_is_posted = TRUE`** — excludes draft/pending entries
3. **Group by `top_level_parent_customer_name` not `customer_id`** — Disney has 3 subsidiary IDs
4. **Labour rows have NULL `service_line_name` and `sub_service_line_name`** — they only appear in the `(blank)` row in any service line breakdown

In [62]:
# Full column schema
sql("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'core'
    AND table_name = 'rpt_project_revenue_and_costs'
    ORDER BY ordinal_position
""")

,column_name,data_type,is_nullable
0,transaction_id,bigint,YES
1,transaction_number,character varying,YES
2,transaction_type,character varying,YES
3,transaction_line_unique_key,bigint,YES
4,transaction_document_number,character varying,YES
5,start_date,timestamp with time zone,YES
6,end_date,timestamp with time zone,YES
7,project_id,bigint,YES
8,project_number,character varying,YES
9,project_name,character varying,YES


In [63]:
# Confirm exactly three account_type_id values exist
# NaN = labour/timesheet rows (786k rows, $34.6M in 2025)
# COGS = direct project costs (39k rows, $45.2M in 2025)
# Income = revenue (6.5k rows, -$135.7M in 2025 — negative because credits)
sql("""
    SELECT
        account_type_id,
        is_part_of_income_statement,
        COUNT(*) as row_count,
        ROUND(SUM(amount)::numeric, 0) as total_amount
    FROM core.rpt_project_revenue_and_costs
    WHERE accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1, 2
    ORDER BY 3 DESC
""")

,account_type_id,is_part_of_income_statement,row_count,total_amount
0,NaN,False,786520,34578815.0
1,COGS,True,39882,45190818.0
2,Income,True,6572,-135695510.0
3,Income,False,23,0.0


In [64]:
# Verify amount = amount_usd (confirmed identical — safe to use either)
# We use amount because it matches the Power Pivot DAX model
sql("""
    SELECT
        -ROUND(SUM(amount)::numeric, 0)     AS revenue_amount,
        -ROUND(SUM(amount_usd)::numeric, 0) AS revenue_amount_usd
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id = 'Income'
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
""")

,revenue_amount,revenue_amount_usd
0,135695510.0,135695510.0


In [65]:
# 2025 total revenue — should equal $135,695,510 (verified against Excel Flash-SL sheet)
df_rev = sql("""
    SELECT
        -SUM(amount) AS total_revenue_2025
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id = 'Income'
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
""")
print(f"2025 Revenue: ${df_rev['total_revenue_2025'][0]/1e6:.1f}M  ✅ matches Excel $135.7M")

2025 Revenue: $135.7M  ✅ matches Excel $135.7M


In [66]:
# Revenue by service line — verified against Excel
sql("""
    SELECT
        COALESCE(service_line_name, '(blank)') AS service_line,
        -ROUND(SUM(amount)::numeric, 0)         AS revenue_2025
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id = 'Income'
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1
    ORDER BY 2 DESC
""")

,service_line,revenue_2025
0,Ad Solutions,83118879.0
1,Custom,24618216.0
2,Brand Tracking,23498047.0
3,Content,3133214.0
4,Data Science,2161222.0
5,(blank),-834068.0


In [67]:
# Labour rows — all have NULL service_line_name and sub_service_line_name
# This is why labour only shows in the (blank) row in any service line breakdown
# Confirmed: 786,520 rows, $34.6M in 2025
sql("""
    SELECT
        COALESCE(service_line_name, 'NULL')     AS service_line,
        COALESCE(sub_service_line_name, 'NULL') AS sub_service_line,
        COUNT(*)                                 AS row_count,
        ROUND(SUM(amount)::numeric, 0)           AS total_amount
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id IS NULL
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1, 2
    ORDER BY 3 DESC
    LIMIT 10
""")

,service_line,sub_service_line,row_count,total_amount
0,NULL,NULL,786520,34578815.0


In [68]:
# Customer discounts — $834k posted to NULL service line
# These are contra-revenue entries (discounts given to clients)
# They appear as negative revenue in the (blank) service line row
# QUESTION FOR JOHN: should these be allocated to specific service lines?
sql("""
    SELECT
        account_name,
        COUNT(*) AS rows,
        ROUND(SUM(amount)::numeric, 0) AS total_amount
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id = 'Income'
    AND service_line_name IS NULL
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1
    ORDER BY 3
""")

,account_name,rows,total_amount
0,Revenues : Customer Discounts,513,834068.0


In [69]:
# Top 10 clients by 2025 revenue — verified against Excel
sql("""
    SELECT
        r.top_level_parent_customer_name AS client,
        -ROUND(SUM(r.amount)::numeric, 0) AS revenue_2025
    FROM core.rpt_project_revenue_and_costs r
    WHERE r.account_type_id = 'Income'
    AND r.accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM r.accounting_period_start_date) = 2025
    AND r.top_level_parent_customer_name IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""")

,client,revenue_2025
0,NBCUniversal,16297212.0
1,Disney,12481249.0
2,Meta,9400637.0
3,Ford Motor Corp,7216524.0
4,LEGO,6533259.0
5,Amazon,5311251.0
6,Google,4136333.0
7,Paramount Global,3931904.0
8,Walmart,3393091.0
9,Chase,3387886.0


---
## 3. `core.dim_customers` — Customer Hierarchy

### What it is
A dimension table mapping individual customer IDs to their top-level parent. Essential for consolidating subsidiary companies into one client row.

### Why it matters
Disney has 3 customer IDs in the fact table:
- `Disney: The Walt Disney Company` (customer_id 44119) — $518,963 BE revenue
- `Disney` (customer_id 4355) — $499,250 BE revenue  
- `Disney: Hulu` (customer_id 4769) — $40,000 BE revenue

Without this join, Disney would appear three times in every client breakdown. With the join, all three consolidate into one `Disney` row.

### Join key
`rpt_project_revenue_and_costs.customer_id` → `dim_customers.customer_id`

In [70]:
# dim_customers schema
sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'core'
    AND table_name = 'dim_customers'
    ORDER BY ordinal_position
""")

,column_name,data_type
0,customer_id,bigint
1,entity_id,character varying
2,customer_external_id,character varying
3,customer_name,character varying
4,customer_full_name,character varying
5,is_person,boolean
6,first_name,character varying
7,last_name,character varying
8,email_address,character varying
9,phone_number,character varying


In [71]:
# Disney subsidiary example — 3 customer_ids, 1 top_level_parent
# This is why we MUST group by top_level_parent_customer_name not customer_id
sql("""
    SELECT
        c.top_level_parent_customer_name,
        r.customer_id,
        c.customer_full_name,
        -ROUND(SUM(r.amount)::numeric, 0) AS revenue_2025
    FROM core.rpt_project_revenue_and_costs r
    LEFT JOIN core.dim_customers c ON r.customer_id = c.customer_id
    WHERE r.account_type_id = 'Income'
    AND r.accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM r.accounting_period_start_date) = 2025
    AND c.top_level_parent_customer_name = 'Disney'
    GROUP BY 1, 2, 3
    ORDER BY 4 DESC
""")

,top_level_parent_customer_name,customer_id,customer_full_name,revenue_2025
0,Disney,4355,Disney,7889375.0
1,Disney,4769,Disney : Hulu,2016575.0
2,Disney,141658,Disney : Disney Branded Television (DBT),828356.0
3,Disney,128877,Disney : Disney+,586580.0
4,Disney,44119,Disney : The Walt Disney Company,518963.0
5,Disney,124721,Disney : Searchlight Pictures,297763.0
6,Disney,92851,Disney : Walt Disney Company US,200042.0
7,Disney,93361,Disney : ABC,94345.0
8,Disney,5353,Disney : Disney Branded Television (DBT) : Dis...,49250.0


In [72]:
# How many revenue rows have no customer match?
# These show as 'Unassigned' in the dashboard
sql("""
    SELECT
        CASE WHEN c.top_level_parent_customer_name IS NULL
             THEN 'No match' ELSE 'Has match' END AS join_status,
        COUNT(*) AS rows,
        -ROUND(SUM(r.amount)::numeric, 0) AS revenue
    FROM core.rpt_project_revenue_and_costs r
    LEFT JOIN core.dim_customers c ON r.customer_id = c.customer_id
    WHERE r.account_type_id = 'Income'
    AND r.accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM r.accounting_period_start_date) = 2025
    GROUP BY 1
""")

,join_status,rows,revenue
0,Has match,6264,136035263.0
1,No match,331,-339753.0


In [73]:
sql("""
    with base as (SELECT
        CASE WHEN c.top_level_parent_customer_name IS NULL
             THEN 'No match' ELSE 'Has match' END AS join_status,
        *

    FROM core.rpt_project_revenue_and_costs r
    LEFT JOIN core.dim_customers c ON r.customer_id = c.customer_id
    WHERE r.account_type_id is null
    AND r.accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM r.accounting_period_start_date) = 2025 )
    --select join_status, role_name_at_timesheet_date, sum(amount) from base group by 1,2
    select * from base limit 10

""")

,join_status,transaction_id,transaction_number,transaction_type,transaction_line_unique_key,transaction_document_number,start_date,end_date,project_id,project_number,...,top_level_parent_customer_id,top_level_parent_customer_name,is_leaf,is_parent,customer_id_levels_down_from_top,pmi_client,pmi_client_summary,pmi_client_parent,top_15,is_in_mapping_file
0,Has match,None,None,None,None,None,None,None,142651,PRJ55986,...,4355,Disney,0,1,1,Disney,Disney,Disney,Disney,1.0
1,Has match,None,None,None,None,None,None,None,142651,PRJ55986,...,4355,Disney,0,1,1,Disney,Disney,Disney,Disney,1.0
2,Has match,None,None,None,None,None,None,None,142651,PRJ55986,...,4355,Disney,0,1,1,Disney,Disney,Disney,Disney,1.0
3,Has match,None,None,None,None,None,None,None,142651,PRJ55986,...,4355,Disney,0,1,1,Disney,Disney,Disney,Disney,1.0
4,Has match,None,None,None,None,None,None,None,142367,PRJ55731,...,124678,Netflix,0,1,1,Netflix,Netflix,Netflix,Netflix,1.0
5,Has match,None,None,None,None,None,None,None,142367,PRJ55731,...,124678,Netflix,0,1,1,Netflix,Netflix,Netflix,Netflix,1.0
6,Has match,None,None,None,None,None,None,None,142367,PRJ55731,...,124678,Netflix,0,1,1,Netflix,Netflix,Netflix,Netflix,1.0
7,Has match,None,None,None,None,None,None,None,142367,PRJ55731,...,124678,Netflix,0,1,1,Netflix,Netflix,Netflix,Netflix,1.0
8,Has match,None,None,None,None,None,None,None,142565,PRJ55910,...,4355,Disney,1,0,2,NaN,NaN,NaN,NaN,NaN
9,Has match,None,None,None,None,None,None,None,142565,PRJ55910,...,4355,Disney,1,0,2,NaN,NaN,NaN,NaN,NaN


---
## 4. Labour — Two Sources, One Question

### The problem
Labour in the dashboard currently comes from the `NULL` rows in `rpt_project_revenue_and_costs` — these are timesheet allocations that NetSuite pushes into the P&L. They total **$34.6M for 2025**.

However, `core.fact_sd_timesheet_cost` is a **separate, more granular** source of people costs direct from Screendragon (the resource management tool). This table has **daily timesheet entries** with role names, enabling breakdown by:
- **Research Labor** — VP, SVP, Director, Manager roles
- **Ops Labor** — Field, QA, Programming, Ops, Dashboard roles
- **Other Labor** — everything else

### The open question
Do the two sources agree? If `fact_sd_timesheet_cost` totals match the NULL rows in `rpt_project_revenue_and_costs`, we can use the Screendragon data to break down labour by type. If they don't match, we need to understand why.

### Key columns in `fact_sd_timesheet_cost`

| Column | Description |
|---|---|
| `netsuite_project_id` | Joins to `project_id` in the fact table |
| `accounting_period_id` | Joins to `accounting_period_id` in the fact table |
| `timesheet_external_cost` | Cost at billable rate — **this is the labour cost figure** |
| `timesheet_internal_cost` | Cost at internal rate — lower, for internal reporting |
| `actual_hours` | Hours logged |
| `role_name_at_timesheet_date` | Person's role at time of entry — enables labour type classification |

In [74]:
# fact_sd_timesheet_cost schema
sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'core'
    AND table_name = 'fact_sd_timesheet_cost'
    ORDER BY ordinal_position
""")

,column_name,data_type
0,netsuite_project_id,bigint
1,sd_project_id,character varying
2,netsuite_project_number,character varying
3,user_id,integer
4,category,character varying
5,actual_hours,double precision
6,timesheet_date,timestamp without time zone
7,user_first_name,character varying
8,user_last_name,character varying
9,role_name_at_timesheet_date,character varying


In [75]:
# SOURCE 1: Labour from rpt_project_revenue_and_costs (NULL account_type_id rows)
# This is what the dashboard currently uses
df_lab1 = sql("""
    SELECT
        EXTRACT(YEAR FROM accounting_period_start_date)::int AS yr,
        ROUND(SUM(amount)::numeric, 0) AS labour_from_rpt
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id IS NULL
    AND accounting_period_is_posted = TRUE
    GROUP BY 1
    ORDER BY 1
""")
print("Labour from rpt_project_revenue_and_costs:")
print(df_lab1)

Labour from rpt_project_revenue_and_costs:
     yr  labour_from_rpt
0  2024       26008304.0
1  2025       34578815.0
2  2026        7581151.0


In [76]:
# SOURCE 2: Labour from fact_sd_timesheet_cost (Screendragon)
# This is more granular — has role names for classification
df_lab2 = sql("""
    SELECT
        EXTRACT(YEAR FROM accounting_period_start_date)::int AS yr,
        ROUND(SUM(timesheet_external_cost)::numeric, 0) AS labour_external,
        ROUND(SUM(timesheet_internal_cost)::numeric, 0) AS labour_internal,
        ROUND(SUM(actual_hours)::numeric, 1)             AS total_hours
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
    GROUP BY 1
    ORDER BY 1
""")
print("Labour from fact_sd_timesheet_cost:")
print(df_lab2)

Labour from fact_sd_timesheet_cost:
     yr  labour_external  labour_internal  total_hours
0  2024       75206371.0       26008304.0     474788.5
1  2025      100156045.0       34578815.0     599415.4
2  2026       21932845.0        7581151.0     134799.7


In [77]:
# COMPARISON: Do the two sources agree?
# Merge on year and compare
df_compare = df_lab1.merge(df_lab2, on='yr', how='outer')
df_compare['diff_external'] = df_compare['labour_from_rpt'] - df_compare['labour_external']
df_compare['diff_internal'] = df_compare['labour_from_rpt'] - df_compare['labour_internal']
print("Labour comparison — rpt vs timesheet:")
print(df_compare[['yr','labour_from_rpt','labour_external','labour_internal',
                   'diff_external','diff_internal']].to_string(index=False))
print("\nNote: if diff_external is close to zero, external rate is the right column to use")
print("If diff_internal is closer to zero, internal rate is the right column")

Labour comparison — rpt vs timesheet:
  yr  labour_from_rpt  labour_external  labour_internal  diff_external  diff_internal
2024       26008304.0       75206371.0       26008304.0    -49198067.0            0.0
2025       34578815.0      100156045.0       34578815.0    -65577230.0            0.0
2026        7581151.0       21932845.0        7581151.0    -14351694.0            0.0

Note: if diff_external is close to zero, external rate is the right column to use
If diff_internal is closer to zero, internal rate is the right column


In [78]:
# Labour classification by role type — Screendragon data
# Research Labor = senior/strategic roles (VP, SVP, Director, Manager)
# Ops Labor = delivery/field roles (Field, QA, Programming, Ops, Dashboard)
# Other Labor = everything else (analysts, coordinators etc.)
sql("""
    SELECT
        EXTRACT(YEAR FROM accounting_period_start_date)::int AS yr,
        CASE
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%VP%','%SVP%','%Director%','%Manager%'])
            THEN 'Research Labor'
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%Field%','%QA%','%Programming%','%Ops%','%Dashboard%'])
            THEN 'Ops Labor'
            ELSE 'Other Labor'
        END AS labor_type,
        ROUND(SUM(timesheet_external_cost)::numeric, 0) AS labour_cost,
        ROUND(SUM(actual_hours)::numeric, 0)             AS total_hours
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
    GROUP BY 1, 2
    ORDER BY 1, 3 DESC
""")

,yr,labor_type,labour_cost,total_hours
0,2024,Research Labor,41713392.0,178028.0
1,2024,Other Labor,18399093.0,186587.0
2,2024,Ops Labor,15093886.0,110174.0
3,2025,Research Labor,43330018.0,187836.0
4,2025,Ops Labor,36409531.0,265763.0
5,2025,Other Labor,20416496.0,145817.0
6,2026,Ops Labor,10432769.0,76152.0
7,2026,Research Labor,8496728.0,36746.0
8,2026,Other Labor,3003348.0,21902.0


In [79]:
# Top 20 roles by cost — validate the classification logic
# Check if any big roles are falling into 'Other Labor' unexpectedly
sql("""
    SELECT
        role_name_at_timesheet_date,
        CASE
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%VP%','%SVP%','%Director%','%Manager%'])
            THEN 'Research Labor'
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%Field%','%QA%','%Programming%','%Ops%','%Dashboard%'])
            THEN 'Ops Labor'
            ELSE 'Other Labor'
        END AS labor_type,
        ROUND(SUM(timesheet_external_cost)::numeric, 0) AS total_cost,
        ROUND(SUM(actual_hours)::numeric, 0)             AS total_hours
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1, 2
    ORDER BY 3 DESC
    LIMIT 20
""")

,role_name_at_timesheet_date,labor_type,total_cost,total_hours
0,NaN,Other Labor,NaN,4877.0
1,Research Director,Research Labor,16799319.0,69133.0
2,Research Manager,Research Labor,15302500.0,87945.0
3,Research Analyst,Other Labor,11748730.0,85757.0
4,Engineer,Other Labor,8667767.0,49815.0
5,Research VP,Research Labor,8124609.0,24846.0
6,Ops - Data Processing (MK),Ops Labor,3347773.0,24436.0
7,Ops - Ad Editorial,Ops Labor,3160563.0,23070.0
8,Research SVP,Research Labor,3103590.0,5912.0
9,Ops - Programming (DM),Ops Labor,2565380.0,18725.0


In [80]:
# Can we join labour from Screendragon to the revenue fact table by project?
# This would allow us to attribute labour to service lines
# The join key is: fact_sd_timesheet_cost.netsuite_project_id = rpt_project_revenue_and_costs.project_id
sql("""
    SELECT
        COUNT(DISTINCT t.netsuite_project_id) AS timesheet_projects,
        COUNT(DISTINCT r.project_id)           AS revenue_projects,
        COUNT(DISTINCT t.netsuite_project_id)
            FILTER (WHERE t.netsuite_project_id IN (
                SELECT DISTINCT project_id
                FROM core.rpt_project_revenue_and_costs
                WHERE EXTRACT(YEAR FROM accounting_period_start_date) = 2025
            )) AS matching_projects
    FROM core.fact_sd_timesheet_cost t
    CROSS JOIN (
        SELECT DISTINCT project_id
        FROM core.rpt_project_revenue_and_costs
        WHERE EXTRACT(YEAR FROM accounting_period_start_date) = 2025
        LIMIT 1
    ) r
    WHERE EXTRACT(YEAR FROM t.accounting_period_start_date) = 2025
""")

,timesheet_projects,revenue_projects,matching_projects
0,1168,1,1168


In [81]:
# If the project join works, we can attribute labour to service lines
# This is the key question — can we break the (blank) labour row into service lines?
sql("""
    SELECT
        COALESCE(r.service_line_name, '(blank)') AS service_line,
        CASE
            WHEN t.role_name_at_timesheet_date ILIKE ANY(ARRAY['%VP%','%SVP%','%Director%','%Manager%'])
            THEN 'Research Labor'
            WHEN t.role_name_at_timesheet_date ILIKE ANY(ARRAY['%Field%','%QA%','%Programming%','%Ops%','%Dashboard%'])
            THEN 'Ops Labor'
            ELSE 'Other Labor'
        END AS labor_type,
        ROUND(SUM(t.timesheet_external_cost)::numeric, 0) AS labour_cost
    FROM core.fact_sd_timesheet_cost t
    LEFT JOIN (
        SELECT DISTINCT project_id, service_line_name
        FROM core.rpt_project_revenue_and_costs
        WHERE accounting_period_is_posted = TRUE
        AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    ) r ON t.netsuite_project_id = r.project_id
    WHERE t.accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM t.accounting_period_start_date) = 2025
    GROUP BY 1, 2
    ORDER BY 3 DESC
""")

,service_line,labor_type,labour_cost
0,(blank),Research Labor,43330018.0
1,(blank),Ops Labor,36409531.0
2,Ad Solutions,Ops Labor,21796844.0
3,(blank),Other Labor,20416496.0
4,Ad Solutions,Research Labor,19438509.0
5,Custom,Research Labor,10990830.0
6,Brand Tracking,Research Labor,10985547.0
7,Brand Tracking,Ops Labor,8224720.0
8,Ad Solutions,Other Labor,5124221.0
9,Custom,Ops Labor,4108472.0


In [82]:
# Labour split by client and service line
# Uses fact_sd_timesheet_cost (internal cost) joined to rpt_project_revenue_and_costs
# The join key is: netsuite_project_id = project_id + accounting_period_id = accounting_period_id
# Pre-aggregate timesheet to project/period grain BEFORE joining to avoid fan-out inflation

sql( '''
WITH labour_by_project AS (
    -- Step 1: aggregate timesheet to project + period level
    -- CRITICAL: must do this before joining to the fact table
    -- otherwise each timesheet row gets multiplied by the number
    -- of transaction lines for that project/period
    SELECT
        netsuite_project_id,
        accounting_period_id,
        SUM(timesheet_internal_cost) AS labour_cost,
        SUM(actual_hours)            AS total_hours
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
    GROUP BY 1, 2
),

revenue_base AS (
    -- Step 2: get one row per project/period with its dimensions
    -- Use MAX to collapse multiple transaction lines into one row per project/period
    SELECT DISTINCT
        r.project_id,
        r.accounting_period_id,
        r.accounting_period_name,
        r.accounting_period_start_date,
        r.service_line_name,
        r.sub_service_line_name,
        r.vertical_name,
        c.top_level_parent_customer_name
    FROM core.rpt_project_revenue_and_costs r
    LEFT JOIN core.dim_customers c ON r.customer_id = c.customer_id
    WHERE r.accounting_period_is_posted = TRUE
)

SELECT
    EXTRACT(YEAR FROM rb.accounting_period_start_date)::int AS yr,
    rb.accounting_period_name,
    rb.accounting_period_start_date,
    COALESCE(rb.top_level_parent_customer_name, 'Unassigned') AS top_level_parent_customer_name,
    COALESCE(rb.service_line_name, '(blank)')                 AS service_line_name,
    COALESCE(rb.sub_service_line_name, '(blank)')             AS sub_service_line_name,
    COALESCE(rb.vertical_name, '(blank)')                     AS vertical_name,
    ROUND(SUM(l.labour_cost)::numeric, 2)                     AS labour_cost,
    ROUND(SUM(l.total_hours)::numeric, 1)                     AS total_hours
FROM labour_by_project l
JOIN revenue_base rb
    ON  l.netsuite_project_id  = rb.project_id
    AND l.accounting_period_id = rb.accounting_period_id
GROUP BY 1, 2, 3, 4, 5, 6, 7
ORDER BY rb.accounting_period_start_date, labour_cost DESC '''
)

,yr,accounting_period_name,accounting_period_start_date,top_level_parent_customer_name,service_line_name,sub_service_line_name,vertical_name,labour_cost,total_hours
0,2024,Jan 2024,2024-01-01,Intuit,Ad Solutions,Brand Effect,Financial Services,NaN,5.0
1,2024,Jan 2024,2024-01-01,Citibank,Ad Solutions,Brand Effect,Financial Services,NaN,0.0
2,2024,Jan 2024,2024-01-01,Pfizer,Custom,Custom,Health,NaN,0.0
3,2024,Jan 2024,2024-01-01,Benjamin Moore & Co.,(blank),(blank),(blank),NaN,3.0
4,2024,Jan 2024,2024-01-01,Tiger Balm,(blank),(blank),(blank),NaN,0.0
...,...,...,...,...,...,...,...,...,...
8762,2026,Apr 2026,2026-04-01,US Cellular,Ad Solutions,Brand Effect,Telco & Cable,0.0,0.0
8763,2026,Apr 2026,2026-04-01,Wells Fargo,Ad Solutions,Ad Effect,Financial Services,0.0,0.0
8764,2026,Apr 2026,2026-04-01,Weber-Stephen Products,(blank),(blank),(blank),0.0,0.0
8765,2026,Apr 2026,2026-04-01,US Cellular,(blank),(blank),(blank),0.0,0.0


In [83]:
df_lab_split = sql("""
WITH project_dimensions AS (
    -- One row per project — pick the dominant service line (highest revenue)
    SELECT DISTINCT ON (project_id)
        project_id,
        service_line_name,
        sub_service_line_name,
        vertical_name,
        customer_id
    FROM core.rpt_project_revenue_and_costs
    WHERE accounting_period_is_posted = TRUE
    AND account_type_id = 'Income'
    ORDER BY project_id, amount ASC
),

periods AS (
    -- Period metadata — join by accounting_period_id
    SELECT DISTINCT
        accounting_period_id,
        accounting_period_name,
        accounting_period_start_date
    FROM core.rpt_project_revenue_and_costs
    WHERE accounting_period_is_posted = TRUE
)

SELECT
    EXTRACT(YEAR FROM p.accounting_period_start_date)::int    AS yr,
    p.accounting_period_name,
    p.accounting_period_start_date,
    COALESCE(c.top_level_parent_customer_name, 'Unassigned')  AS top_level_parent_customer_name,
    COALESCE(d.service_line_name, '(blank)')                  AS service_line_name,
    COALESCE(d.sub_service_line_name, '(blank)')              AS sub_service_line_name,
    COALESCE(d.vertical_name, '(blank)')                      AS vertical_name,
    ROUND(SUM(t.timesheet_internal_cost)::numeric, 2)         AS labour_cost,
    ROUND(SUM(t.actual_hours)::numeric, 1)                    AS total_hours
FROM core.fact_sd_timesheet_cost t
LEFT JOIN project_dimensions d  ON t.netsuite_project_id  = d.project_id
LEFT JOIN core.dim_customers c  ON d.customer_id          = c.customer_id
LEFT JOIN periods p             ON t.accounting_period_id = p.accounting_period_id
WHERE t.accounting_period_is_posted = TRUE
GROUP BY 1, 2, 3, 4, 5, 6, 7
ORDER BY p.accounting_period_start_date, labour_cost DESC
""")

print("2025 labour by service line:")
print(
    df_lab_split[df_lab_split['yr'] == 2025]
    .groupby('service_line_name')['labour_cost']
    .sum().sort_values(ascending=False).round(0)
)
print(f"\nTotal 2025: ${df_lab_split[df_lab_split['yr'] == 2025]['labour_cost'].sum()/1e6:.1f}M")
print("(expect ~$34.6M)")

2025 labour by service line:
service_line_name
(blank)           10817806.0
Ad Solutions      10392483.0
Brand Tracking     6950864.0
Custom             5184078.0
Content             684056.0
Data Science        549528.0
Name: labour_cost, dtype: float64

Total 2025: $34.6M
(expect ~$34.6M)


In [84]:
sql("""
select distinct  EXTRACT(YEAR FROM accounting_period_start_date)::int    AS yr from core.fact_sd_timesheet_cost order by 1 desc

""")

,yr
0,2026
1,2025
2,2024


---
## 5. Gross Margin Calculation

### The allocation methodology
Three product lines have fixed yearly overhead costs that are allocated across clients proportionally:

| Product | Sub service line filter | 2025 fixed cost |
|---|---|---|
| Brand Effect | `sub_service_line_name = 'Brand Effect'` | $10,158,000 |
| AE Syndicated | `sub_service_line_name = 'Ad Effect'` | $938,000 |
| Ad Solutions RTA | `sub_service_line_name = 'Ad Solutions RTA'` | $833,000 |

### The formula
```
Monthly allocation = (client annual BE revenue / total annual BE revenue) × fixed_cost / 12
```

### Design decisions
- **Fan out to all 12 months** — fixed cost exists regardless of client activity in any given month
- **Group by `top_level_parent_customer_name`** — otherwise Disney appears 3 times
- **FULL OUTER JOIN** in the final query — some clients have allocations but no base revenue rows for a given month
- **No rounding at row level** — accumulates penny errors; round only at display

In [85]:
# Verify BE allocation sums to $10,158,000 for 2025
# This tests the core allocation logic
sql("""
    WITH fixed_costs AS (
        SELECT 2025 AS yr, 10158000 AS fixed_cost_be
    ),
    be_rev AS (
        SELECT
            EXTRACT(YEAR FROM r.accounting_period_start_date)::int AS yr,
            COALESCE(c.top_level_parent_customer_name, 'Unassigned') AS customer,
            -SUM(r.amount) AS annual_be_rev
        FROM core.rpt_project_revenue_and_costs r
        LEFT JOIN core.dim_customers c ON r.customer_id = c.customer_id
        WHERE r.account_type_id = 'Income'
        AND r.sub_service_line_name = 'Brand Effect'
        AND r.accounting_period_is_posted = TRUE
        AND EXTRACT(YEAR FROM r.accounting_period_start_date) = 2025
        GROUP BY 1, 2
    ),
    tot AS (SELECT yr, SUM(annual_be_rev) AS tot FROM be_rev GROUP BY 1),
    periods AS (
        SELECT DISTINCT
            EXTRACT(YEAR FROM accounting_period_start_date)::int AS yr,
            accounting_period_name
        FROM core.rpt_project_revenue_and_costs
        WHERE accounting_period_is_posted = TRUE
        AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    )
    SELECT
        SUM(
            (b.annual_be_rev / NULLIF(t.tot, 0)) * f.fixed_cost_be / 12
        ) AS total_be_allocation
    FROM be_rev b
    JOIN tot t ON b.yr = t.yr
    JOIN fixed_costs f ON b.yr = f.yr
    CROSS JOIN periods p
""")

,total_be_allocation
0,10158000.0


In [86]:
# Load full gross margin query and check 2025 totals
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

QUERIES_DIR = Path('/Users/sihamchowdhury/Desktop/Projects/finance-dashboards/src/queries')

with engine.connect() as conn:
    df_gm = pd.read_sql(
        text((QUERIES_DIR / '5_gross_margin.sql').read_text()),
        conn
    )

df_gm['accounting_period_start_date'] = pd.to_datetime(df_gm['accounting_period_start_date'])
df_gm['contribution'] = df_gm['gross_margin'] - df_gm['labour']

df_2025 = df_gm[df_gm['accounting_period_start_date'].dt.year == 2025]

totals = df_2025[['revenue','cogs','labour','be_allocation','ae_allocation','rta_allocation','gross_margin']].sum()
totals['contribution'] = totals['gross_margin'] - totals['labour']
totals['gm_pct'] = totals['gross_margin'] / totals['revenue'] * 100
totals['cm_pct'] = totals['contribution'] / totals['revenue'] * 100

print("=== 2025 P&L Summary ===")
for k, v in totals.items():
    if 'pct' in k:
        print(f"{k:25s}: {v:.1f}%")
    else:
        print(f"{k:25s}: ${v/1e6:.1f}M")

=== 2025 P&L Summary ===
revenue                  : $135.7M
cogs                     : $45.2M
labour                   : $34.6M
be_allocation            : $10.2M
ae_allocation            : $0.9M
rta_allocation           : $0.8M
gross_margin             : $78.6M
contribution             : $44.0M
gm_pct                   : 57.9%
cm_pct                   : 32.4%


In [87]:
# P&L by service line — matches Excel Contribution sheet
# Note: Brand Effect GM% is 8% because of the large BE allocation ($10.2M)
gm_sl = df_2025.groupby('service_line_name', dropna=False).agg(
    revenue=('revenue','sum'),
    cogs=('cogs','sum'),
    be_allocation=('be_allocation','sum'),
    ae_allocation=('ae_allocation','sum'),
    rta_allocation=('rta_allocation','sum'),
    gross_margin=('gross_margin','sum'),
    labour=('labour','sum')
).reset_index()

gm_sl['contribution'] = gm_sl['gross_margin'] - gm_sl['labour']
gm_sl['gm_pct'] = (gm_sl['gross_margin'] / gm_sl['revenue'].replace(0,float('nan')) * 100).round(1)
gm_sl['cm_pct'] = (gm_sl['contribution'] / gm_sl['revenue'].replace(0,float('nan')) * 100).round(1)

# Display in $000s to match Excel
for col in ['revenue','cogs','be_allocation','ae_allocation','rta_allocation','gross_margin','labour','contribution']:
    gm_sl[col] = (gm_sl[col]/1000).round(0)

gm_sl.sort_values('revenue', ascending=False)

,service_line_name,revenue,cogs,be_allocation,ae_allocation,rta_allocation,gross_margin,labour,contribution,gm_pct,cm_pct
0,Ad Solutions,83119.0,27602.0,10158.0,938.0,833.0,43588.0,0.0,43588.0,52.4,52.4
3,Custom,24618.0,9425.0,0.0,0.0,0.0,15193.0,0.0,15193.0,61.7,61.7
1,Brand Tracking,23498.0,7132.0,0.0,0.0,0.0,16366.0,0.0,16366.0,69.6,69.6
2,Content,3133.0,890.0,0.0,0.0,0.0,2243.0,0.0,2243.0,71.6,71.6
4,Data Science,2161.0,473.0,0.0,0.0,0.0,1689.0,0.0,1689.0,78.1,78.1
5,NaN,-834.0,-331.0,0.0,0.0,0.0,-503.0,34579.0,-35082.0,60.3,4206.1


---
## 6. Interactive Query Function

Use `get_metrics()` to slice the gross margin data by any combination of dimensions.

In [88]:
def get_metrics(
    year: int = 2025,
    sub_service_line: str = None,
    service_line: str = None,
    customer: str = None,
    vertical: str = None,
    group_by: list = None
) -> pd.DataFrame:
    """
    Slice gross margin data by any combination of dimensions.

    Parameters
    ----------
    year : int
        Accounting year (default 2025)
    sub_service_line : str, optional
        Filter to sub service line e.g. 'Brand Effect'
    service_line : str, optional
        Filter to service line e.g. 'Ad Solutions'
    customer : str, optional
        Filter to top level parent customer e.g. 'Disney'
    vertical : str, optional
        Filter to vertical e.g. 'TV'
    group_by : list, optional
        Columns to group by. Default: ['sub_service_line_name']
        Options: 'service_line_name', 'sub_service_line_name',
                 'top_level_parent_customer_name', 'vertical_name',
                 'accounting_period_name'

    Returns
    -------
    pd.DataFrame with all P&L metrics
    """
    if group_by is None:
        group_by = ['sub_service_line_name']

    # Apply filters
    mask = df_gm['accounting_period_start_date'].dt.year == year
    if sub_service_line: mask &= df_gm['sub_service_line_name'] == sub_service_line
    if service_line:     mask &= df_gm['service_line_name'] == service_line
    if customer:         mask &= df_gm['top_level_parent_customer_name'] == customer
    if vertical:         mask &= df_gm['vertical_name'] == vertical

    df = df_gm[mask].copy()
    if df.empty:
        print("No data for given parameters")
        return pd.DataFrame()

    result = df.groupby(group_by).agg(
        revenue=('revenue','sum'),
        cogs=('cogs','sum'),
        labour=('labour','sum'),
        be_allocation=('be_allocation','sum'),
        ae_allocation=('ae_allocation','sum'),
        rta_allocation=('rta_allocation','sum'),
        gross_margin=('gross_margin','sum')
    ).reset_index()

    result['contribution'] = result['gross_margin'] - result['labour']
    result['gm_pct']       = (result['gross_margin'] / result['revenue'].replace(0,float('nan')) * 100).round(1)
    result['cm_pct']       = (result['contribution'] / result['revenue'].replace(0,float('nan')) * 100).round(1)

    for col in ['revenue','cogs','labour','be_allocation','ae_allocation','rta_allocation','gross_margin','contribution']:
        result[col] = result[col].round(0)

    return result.sort_values('revenue', ascending=False)

print("get_metrics() loaded")

get_metrics() loaded


In [89]:
# Example 1: All service lines 2025
get_metrics(year=2025, group_by=['service_line_name'])

,service_line_name,revenue,cogs,labour,be_allocation,ae_allocation,rta_allocation,gross_margin,contribution,gm_pct,cm_pct
0,Ad Solutions,83118879.0,27601541.0,0.0,10158000.0,938000.0,833000.0,43588339.0,43588339.0,52.4,52.4
3,Custom,24618216.0,9425375.0,0.0,0.0,0.0,0.0,15192841.0,15192841.0,61.7,61.7
1,Brand Tracking,23498047.0,7131892.0,0.0,0.0,0.0,0.0,16366155.0,16366155.0,69.6,69.6
2,Content,3133214.0,890241.0,0.0,0.0,0.0,0.0,2242973.0,2242973.0,71.6,71.6
4,Data Science,2161222.0,472658.0,0.0,0.0,0.0,0.0,1688564.0,1688564.0,78.1,78.1


In [90]:
# Example 2: Disney full P&L by sub service line
get_metrics(year=2025, customer='Disney', group_by=['service_line_name','sub_service_line_name'])

,service_line_name,sub_service_line_name,revenue,cogs,labour,be_allocation,ae_allocation,rta_allocation,gross_margin,contribution,gm_pct,cm_pct
3,Ad Solutions,CA,7988750.0,3147242.0,0.0,0.0,0.0,0.0,4841507.0,4841507.0,60.6,60.6
5,Content,Custom,1353470.0,335013.0,0.0,0.0,0.0,0.0,1018457.0,1018457.0,75.2,75.2
2,Ad Solutions,Brand Effect,1058212.0,0.0,0.0,469953.0,0.0,0.0,588260.0,588260.0,55.6,55.6
0,Ad Solutions,Ad Effect,895700.0,165250.0,0.0,0.0,47502.0,0.0,682948.0,682948.0,76.2,76.2
4,Brand Tracking,Brand,598097.0,479623.0,0.0,0.0,0.0,0.0,118474.0,118474.0,19.8,19.8
6,Custom,Custom,547000.0,154050.0,0.0,0.0,0.0,0.0,392950.0,392950.0,71.8,71.8
1,Ad Solutions,Ad Solutions RTA,40020.0,0.0,0.0,0.0,0.0,10335.0,29685.0,29685.0,74.2,74.2


In [91]:
# Example 3: Brand Effect — top 10 clients by allocation
get_metrics(year=2025, sub_service_line='Brand Effect',
            group_by=['top_level_parent_customer_name']).head(10)

,top_level_parent_customer_name,revenue,cogs,labour,be_allocation,ae_allocation,rta_allocation,gross_margin,contribution,gm_pct,cm_pct
30,NBCUniversal,4833350.0,29209.0,0.0,2146493.0,0.0,0.0,2657648.0,2657648.0,55.0,55.0
15,Fox,1828013.0,143.0,0.0,811822.0,0.0,0.0,1016048.0,1016048.0,55.6,55.6
35,Nissan Motor Corp,1420069.0,0.0,0.0,630653.0,0.0,0.0,789415.0,789415.0,55.6,55.6
10,DirecTV,1225000.0,0.0,0.0,544023.0,0.0,0.0,680977.0,680977.0,55.6,55.6
52,Walmart,1085062.0,5680.0,0.0,481877.0,0.0,0.0,597506.0,597506.0,55.1,55.1
44,State Farm,1076910.0,2537.0,0.0,478256.0,0.0,0.0,596117.0,596117.0,55.4,55.4
11,Disney,1058212.0,0.0,0.0,469953.0,0.0,0.0,588260.0,588260.0,55.6,55.6
5,Capital One,896000.0,0.0,0.0,397914.0,0.0,0.0,498086.0,498086.0,55.6,55.6
21,Intuit,823717.0,1923.0,0.0,365813.0,0.0,0.0,455981.0,455981.0,55.4,55.4
29,Miller Coors,740040.0,3583.0,0.0,328652.0,0.0,0.0,407804.0,407804.0,55.1,55.1


In [92]:
# Example 4: Monthly GM% trend for Ad Solutions 2025
get_metrics(year=2025, service_line='Ad Solutions', group_by=['accounting_period_name'])

,accounting_period_name,revenue,cogs,labour,be_allocation,ae_allocation,rta_allocation,gross_margin,contribution,gm_pct,cm_pct
3,Feb 2025,7971310.0,2502249.0,0.0,846500.0,78167.0,69417.0,4474978.0,4474978.0,56.1,56.1
8,May 2025,7749892.0,2847541.0,0.0,846500.0,78167.0,69417.0,3908267.0,3908267.0,50.4,50.4
0,Apr 2025,7554633.0,2730455.0,0.0,846500.0,78167.0,69417.0,3830094.0,3830094.0,50.7,50.7
9,Nov 2025,7115513.0,2261768.0,0.0,846500.0,78167.0,69417.0,3859662.0,3859662.0,54.2,54.2
1,Aug 2025,7091759.0,2169432.0,0.0,846500.0,78167.0,69417.0,3928243.0,3928243.0,55.4,55.4
6,Jun 2025,6917613.0,2131327.0,0.0,846500.0,78167.0,69417.0,3792203.0,3792203.0,54.8,54.8
7,Mar 2025,6723208.0,2428753.0,0.0,846500.0,78167.0,69417.0,3300371.0,3300371.0,49.1,49.1
4,Jan 2025,6662080.0,1978637.0,0.0,846500.0,78167.0,69417.0,3689360.0,3689360.0,55.4,55.4
5,Jul 2025,6476723.0,2307214.0,0.0,846500.0,78167.0,69417.0,3175426.0,3175426.0,49.0,49.0
11,Sep 2025,6425624.0,2381618.0,0.0,846500.0,78167.0,69417.0,3049922.0,3049922.0,47.5,47.5


---
## 7. Pipeline — `core.rpt_hubspot_line_report`

### What it is
HubSpot deal lines — each row is a deal line item with its current pipeline stage, value, service line, vertical, and owner. This is the CRM pipeline view.

### Key things to understand

**Deal vs deal line:** One deal can have multiple lines (one per service line or product). The `deal_id` is the deal-level identifier — use `COUNT(DISTINCT deal_id)` for deal counts, not `COUNT(*)`.

**Multi-service-line deals:** Some deals have `service_line` as a semicolon-separated string like `"Brand Tracking;Campaign Analytics"`. These need to be exploded in Python.

**Stage naming:** The stage names are unusual — some stages are named `Q1 2025`, `Q2 2025` etc. which appear to be custom stages rather than standard funnel stages. The standard funnel is Stage 0 → Stage 6.

**Always exclude Closed Won and Closed Lost** for pipeline views — these are historical, not forward-looking.

### What the pipeline tab shows
- Total active pipeline value
- Breakdown by stage (funnel)
- Breakdown by service line
- Detail table with stage + service line + vertical

In [93]:
# rpt_hubspot_line_report schema
sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'core'
    AND table_name = 'rpt_hubspot_line_report'
    ORDER BY ordinal_position
""")

,column_name,data_type
0,deal_line_id,bigint
1,deal_id,bigint
2,deal_name,character varying
3,deal_pipeline_id,text
4,deal_pipeline_stage_id,text
5,deal_pipeline_stage_name,character varying
6,project_id,bigint
7,netsuite_customer_name,character varying
8,hubspot_service_name,character varying
9,line_amount,double precision


In [94]:
# All pipeline stages — active pipeline only (excl Closed Won/Lost)
# Note the Q1/Q2/Q3/Q4 2025 stage names — custom stages, not standard funnel
sql("""
    SELECT
        deal_pipeline_stage_name,
        COUNT(DISTINCT deal_id) AS deals,
        ROUND(SUM(deal_amount_usd)::numeric, 0) AS total_value
    FROM core.rpt_hubspot_line_report
    WHERE is_deal_deleted = FALSE
    AND LOWER(deal_pipeline_stage_name) NOT IN ('closed won','closed lost')
    AND deal_amount_usd IS NOT NULL
    GROUP BY 1
    ORDER BY 3 DESC
""")

,deal_pipeline_stage_name,deals,total_value
0,Target Account,57,91361500.0
1,0% Prospecting Pipeline,635,57247987.0
2,Q1 2025,24,38550000.0
3,Q2 2025,23,38400000.0
4,Q4 2025,23,32000000.0
5,Q3 2025,22,32000000.0
6,Stage 5 - Contracting,24,26012606.0
7,Stage 2 - Solutioning,66,24500721.0
8,Stage 1 - Qualifying,76,18502061.0
9,Stage 3 - Proposed,65,18269330.0


In [95]:
# Pipeline by service line
# Note: service_line can contain semicolons for multi-service-line deals
# e.g. 'Brand Tracking;Campaign Analytics'
sql("""
    SELECT
        COALESCE(service_line, 'Unassigned') AS service_line,
        COUNT(DISTINCT deal_id) AS deals,
        ROUND(SUM(deal_amount_usd)::numeric, 0) AS total_value
    FROM core.rpt_hubspot_line_report
    WHERE is_deal_deleted = FALSE
    AND LOWER(deal_pipeline_stage_name) NOT IN ('closed won','closed lost')
    AND deal_amount_usd IS NOT NULL
    GROUP BY 1
    ORDER BY 3 DESC
""")

,service_line,deals,total_value
0,Unassigned,316,151116666.0
1,Campaign Analytics,287,73297320.0
2,Custom Research,291,42230966.0
3,Brand Tracking,105,27974318.0
4,Campaign Analytics;Brand Tracking;Custom Research,7,12800000.0
5,Campaign Analytics;Brand Tracking;Content,1,11500000.0
6,Campaign Analytics;Custom Research,19,10716000.0
7,Brand Tracking;Campaign Analytics;Custom Research,2,9100000.0
8,Brand Tracking;Data Science;Campaign Analytics,1,8800000.0
9,Campaign Analytics;Brand Tracking,9,6619000.0


In [96]:
# Check semicolon deals — these need Python explode treatment
df_pipe = sql("""
    SELECT
        deal_id,
        deal_name,
        deal_pipeline_stage_name,
        COALESCE(service_line, 'Unassigned') AS service_line,
        COALESCE(vertical, 'Unassigned') AS vertical,
        owner_full_name,
        deal_amount_usd AS pipeline_value_usd
    FROM core.rpt_hubspot_line_report
    WHERE is_deal_deleted = FALSE
    AND LOWER(deal_pipeline_stage_name) NOT IN ('closed won','closed lost')
    AND deal_amount_usd IS NOT NULL
""")

multi_sl = df_pipe[df_pipe['service_line'].str.contains(';', na=False)]
print(f"Total pipeline rows: {len(df_pipe)}")
print(f"Unique deals: {df_pipe['deal_id'].nunique()}")
print(f"Multi-service-line deals (semicolons): {len(multi_sl)}")
print(f"Total pipeline value: ${df_pipe.drop_duplicates('deal_id')['pipeline_value_usd'].sum()/1e6:.1f}M")
multi_sl[['deal_id','deal_name','service_line','pipeline_value_usd']].head(5)

Total pipeline rows: 1319
Unique deals: 1122
Multi-service-line deals (semicolons): 94
Total pipeline value: $328.3M


,deal_id,deal_name,service_line,pipeline_value_usd
30,53754294581,T-Mobile - TfB AE & Custom Q4 2026 Placeholder,Campaign Analytics;Custom Research,100000.0
31,53754294581,T-Mobile - TfB AE & Custom Q4 2026 Placeholder,Campaign Analytics;Custom Research,100000.0
36,41599627008,T-Mobile - TfB AE Q2 2026 Study,Campaign Analytics;Custom Research,200000.0
37,41599627008,T-Mobile - TfB AE Q2 2026 Study,Campaign Analytics;Custom Research,200000.0
38,41599627008,T-Mobile - TfB AE Q2 2026 Study,Campaign Analytics;Custom Research,200000.0


In [97]:
# Explode multi-service-line deals — each service line gets its own row
# This is what the dashboard does in get_pipeline()
df_pipe_exploded = df_pipe.copy()
df_pipe_exploded['service_line'] = df_pipe_exploded['service_line'].str.split(';')
df_pipe_exploded = df_pipe_exploded.explode('service_line')
df_pipe_exploded['service_line'] = df_pipe_exploded['service_line'].str.strip()

print(f"Rows before explode: {len(df_pipe)}")
print(f"Rows after explode: {len(df_pipe_exploded)}")

# Group correctly — use nunique on deal_id not count
grouped = (
    df_pipe_exploded
    .groupby(['deal_pipeline_stage_name','service_line'])
    .agg(
        num_deals=('deal_id','nunique'),
        pipeline_value=('pipeline_value_usd','sum')
    )
    .reset_index()
    .sort_values('pipeline_value', ascending=False)
)
grouped.head(10)

Rows before explode: 1319
Rows after explode: 1435


,deal_pipeline_stage_name,service_line,num_deals,pipeline_value
45,Target Account,Campaign Analytics,42,75811500.00
44,Target Account,Brand Tracking,29,64981500.00
47,Target Account,Custom Research,33,46180000.00
15,Q1 2025,Unassigned,24,38550000.00
16,Q2 2025,Unassigned,23,38400000.00
18,Q4 2025,Unassigned,23,32000000.00
17,Q3 2025,Unassigned,22,32000000.00
10,0% Prospecting Pipeline,Campaign Analytics,178,31740809.01
9,0% Prospecting Pipeline,Brand Tracking,56,17023500.00
48,Target Account,Data Science,8,16249500.00


In [98]:
# Top 10 deal owners by pipeline value
(
    df_pipe.drop_duplicates('deal_id')
    .groupby('owner_full_name')
    .agg(deals=('deal_id','nunique'), value=('pipeline_value_usd','sum'))
    .reset_index()
    .sort_values('value', ascending=False)
    .head(10)
)

,owner_full_name,deals,value
37,Lena Kasahara,41,3.184375e+07
40,Mara Doran,76,3.077400e+07
14,Danielle Byrd,31,2.835000e+07
9,Christian Dubreuil,67,2.714375e+07
17,Dounia Turrill,83,2.515371e+07
49,Mike Chung,33,2.431500e+07
24,Greg Conlon,97,2.340057e+07
44,Mark Willard,41,2.304360e+07
30,Jon Neary,37,1.656374e+07
5,Ben Carlson,5,1.302500e+07


---
## 8. Targets — `core.fact_hubspot_targets`

### What it is
Quarterly revenue targets by person and team — uploaded from a spreadsheet into HubSpot and synced to the DB via Fivetran.

### Key limitations
1. **Only from Q1 2025 onwards** — no historical targets in the DB. 2024 and earlier targets exist only in Excel.
2. **Team level only** — no breakdown by service line or vertical
3. **2026 budget not loaded** — the 2026 budget lives in the Excel `Budget` and `Budget SL` sheets only

### What we can do with it
- Show quarterly target vs actual revenue by team (for 2025)
- Calculate attainment % per quarter
- Show annual target by team

### What we cannot do yet
- Actual vs Budget by service line (targets not at SL level)
- Any comparison before Q1 2025
- 2026 Actual vs Budget (budget not in DB)

In [99]:
# fact_hubspot_targets schema
sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'core'
    AND table_name = 'fact_hubspot_targets'
    ORDER BY ordinal_position
""")

,column_name,data_type
0,_line,bigint
1,_fivetran_synced,timestamp with time zone
2,team_primary_name,character varying
3,user_first_name,character varying
4,user_last_name,character varying
5,target_amount_usd,bigint
6,quarter_start_date,date
7,quarter_end_date,date


In [100]:
# Full targets table — only 20 rows (5 teams × 4 quarters)
df_tgt = sql("""
    SELECT
        quarter_start_date,
        quarter_end_date,
        team_primary_name,
        SUM(target_amount_usd) AS target_usd
    FROM core.fact_hubspot_targets
    GROUP BY 1, 2, 3
    ORDER BY 1, 3
""")
df_tgt['quarter_start_date'] = pd.to_datetime(df_tgt['quarter_start_date'])
print(f"Total 2025 target: ${df_tgt['target_usd'].sum()/1e6:.1f}M")
print(f"Teams: {df_tgt['team_primary_name'].unique().tolist()}")
df_tgt

Total 2025 target: $140.9M
Teams: ['Brands - CPG', 'Europe', 'FSA', 'MSE', 'Tech - Telco']


,quarter_start_date,quarter_end_date,team_primary_name,target_usd
0,2025-01-01,2025-03-31,Brands - CPG,9900000.0
1,2025-01-01,2025-03-31,Europe,1800000.0
2,2025-01-01,2025-03-31,FSA,13900000.0
3,2025-01-01,2025-03-31,MSE,7450000.0
4,2025-01-01,2025-03-31,Tech - Telco,5500000.0
5,2025-04-01,2025-06-30,Brands - CPG,7800000.0
6,2025-04-01,2025-06-30,Europe,2100000.0
7,2025-04-01,2025-06-30,FSA,6900000.0
8,2025-04-01,2025-06-30,MSE,10900000.0
9,2025-04-01,2025-06-30,Tech - Telco,10700000.0


In [101]:
# Annual target by team
df_tgt.groupby('team_primary_name')['target_usd'].sum().sort_values(ascending=False).reset_index()

,team_primary_name,target_usd
0,Tech - Telco,35600000.0
1,FSA,34600000.0
2,MSE,33550000.0
3,Brands - CPG,29100000.0
4,Europe,8100000.0


In [102]:
# Compare total 2025 target vs actual revenue
# Note: targets are at TEAM level, revenue is at SERVICE LINE level
# No direct apples-to-apples comparison is possible without a team→service line mapping
actual_2025 = df_2025['revenue'].sum()
target_2025 = df_tgt['target_usd'].sum()

print(f"2025 Target:  ${target_2025/1e6:.1f}M")
print(f"2025 Actual:  ${actual_2025/1e6:.1f}M")
print(f"Variance:     ${(actual_2025-target_2025)/1e6:.1f}M")
print(f"Attainment:   {actual_2025/target_2025*100:.1f}%")
print("\nCAVEAT: Targets are at team level (FSA, MSE, Brands-CPG etc.)")
print("Revenue is at service line level (Ad Solutions, Brand Tracking etc.)")
print("A team→service line mapping is needed for proper comparison")

2025 Target:  $140.9M
2025 Actual:  $135.7M
Variance:     $-5.3M
Attainment:   96.3%

CAVEAT: Targets are at team level (FSA, MSE, Brands-CPG etc.)
Revenue is at service line level (Ad Solutions, Brand Tracking etc.)
A team→service line mapping is needed for proper comparison


---
## 9. Data Model Diagram

```
┌─────────────────────────────────────────────────────┐
│         core.rpt_project_revenue_and_costs          │
│                                                     │
│  project_id ──────────────────────────────────┐    │
│  customer_id ──────────────┐                  │    │
│  accounting_period_id ─────┼──────────────┐   │    │
│  account_type_id           │              │   │    │
│  service_line_name         │              │   │    │
│  sub_service_line_name     │              │   │    │
│  vertical_name             │              │   │    │
│  amount (revenue/COGS/lab) │              │   │    │
└────────────────────────────┼──────────────┼───┼────┘
                             │              │   │
                             ▼              │   ▼
┌──────────────────────┐     │    ┌─────────┼──────────────────────┐
│   core.dim_customers │     │    │  core.fact_sd_timesheet_cost   │
│                      │     │    │                                │
│  customer_id ◄───────┘     │    │  netsuite_project_id ◄────────┘
│  customer_full_name        │    │  accounting_period_id ◄────────┘
│  top_level_parent_         │    │  role_name_at_timesheet_date   │
│    customer_name           │    │  timesheet_external_cost       │
│                            │    │  actual_hours                  │
└──────────────────────┘     │    └────────────────────────────────┘

┌───────────────────────────────┐  ┌────────────────────────────────┐
│ core.rpt_hubspot_line_report  │  │  core.fact_hubspot_targets     │
│                               │  │                                │
│  deal_id                      │  │  team_primary_name             │
│  deal_pipeline_stage_name     │  │  quarter_start_date            │
│  deal_amount_usd              │  │  target_amount_usd             │
│  service_line                 │  │                                │
│  vertical                     │  │  ⚠️ Only from Q1 2025          │
│  owner_full_name              │  │  ⚠️ Team level only            │
└───────────────────────────────┘  └────────────────────────────────┘
```

---
## 10. Open Questions for the 29 April Meeting

### For John Benak (Finance)

| # | Question | Why it matters |
|---|---|---|
| 1 | **What is the `have` filter in Excel?** | Causes $30M gap: SQL $135.7M vs Excel $105.1M. Is it a `has_revenue` flag, a specific client filter, or something else? |
| 2 | **Is `timesheet_external_cost` the right column for labour?** | Current dashboard uses NULL rows in `rpt_project_revenue_and_costs` ($34.6M). Screendragon has `external_cost` vs `internal_cost`. |
| 3 | **Should customer discounts ($834k) be allocated to service lines?** | Currently they sit in the `(blank)` row with no service line. |
| 4 | **AE Synd service IDs** — DAX uses `service_id IN (120,127,142,1089)`. Still valid? | Our SQL uses `sub_service_line_name = 'Ad Effect'` instead. Are these equivalent? |
| 5 | **Internal vertical** — should it be included or excluded? | Removing it drops COGS significantly. |

### For Erik Wiedemeijer (VP Data Science)

| # | Question | Why it matters |
|---|---|---|
| 6 | **Confirm fixed cost values for 2026 and beyond** | Currently using 2025 BE = $10.158M. 2026 = $1.015M (placeholder). |
| 7 | **Materialised views** — can CJ create them? | Reduces load time from ~10s to <1s. Two views needed: `mv_gross_margin` and `mv_labour_by_type`. |
| 8 | **Azure App Service deployment** — can CJ prioritise? | Needed for production hosting with Azure AD auth. |

### For Paul Forgue (CFO)

| # | Question | Why it matters |
|---|---|---|
| 9 | **Tooling decision** — Power BI or Streamlit on Azure? | Julian suggested Streamlit saves PBI licensing costs. Need Paul's steer. |
| 10 | **2026 budget** — can it be loaded into the DB? | Currently Excel only. Needed for Actual vs Budget view. |
| 11 | **Target→Service line mapping** — does one exist? | Needed to compare targets (team level) vs actuals (service line level). |

### For Julian Highley (EVP Data Science)

| # | Question | Why it matters |
|---|---|---|
| 12 | **Governance sign-off on Azure App Service + AD auth plan** | Julian raised concerns about Streamlit Cloud. Azure App Service resolves all concerns — needs his sign-off. |